In [5]:
import logging

logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s %(levelname)-8s %(name)s: %(message)s',
    datefmt='%H:%M:%S',  # %Y-%M-%D %H:%M:%S
    force=True
)

# 이름을 붙여 로거를 하나 가져온다
log=logging.getLogger('demo')
log.debug('It is a debug with the debug mode.')
log.info('It is a log of an info level.')
log.warning('Be aware. It is a warning.')
log.error('An error occured.')

# 로그 포맷 패턴
# 시간, 로그 레벨, 로거 이름, 메시지(내용)
# %(asctime)s, %(levelname)s, %(name)s, %(message)s

14:39:21 DEBUG    demo: It is a debug with the debug mode.
14:39:21 INFO     demo: It is a log of an info level.
14:39:21 WARNING  demo: Be aware. It is a warning.
14:39:21 ERROR    demo: An error occured.


In [6]:
import sys, logging
from pathlib import Path

backend_path = str(Path.cwd().parents[2] / 'backend')  # 저장소 루트의 backend 절대경로
if backend_path not in sys.path:
    sys.path.insert(0, backend_path)

# 앞 셀에서 루트 로거를 건드렸으면, 아래와 같이 깨끗이 하고 새로 얹는다.
logging.getLogger().handlers=[]
for m in [k for k in list(sys.modules) if k.startswith('app.core.logging')]:
    del sys.modules[m]

# ---------------------------------------
from app.core.logging import get_logger

logger=get_logger('app.services.document') # 임시 패키지명
logger.info('문서 적재 완료: %s', 'DOC-HR-001') # f'' -> %s 권장
logger.warning('임계치에 근접합니다. 주의하세요.')

14:39:21 INFO     app.services.document: 문서 적재 완료: DOC-HR-001
14:39:21 WARNING  app.services.document: 임계치에 근접합니다. 주의하세요.


In [7]:
class AgentError(Exception):
    status_code, code=400, 'agent_error'
    def __init__(self, message, *, detail=None):
        super().__init__(message)
        self.message, self.detail = message, detail

class NotFound(AgentError):
    status_code, code=404, 'not_found'

class ExternalServiceError(AgentError):
    status_code, code=502, 'external_error'

def fake_parsing(doc_id):
    if doc_id=='DOC-BAD':
        raise ConnectionError('Bad Connection: api.llm.ai:443')
    return f'{doc_id}의 본문 텍스트'

def ingest(doc_id):
    logger.info('적재 시작: %s', doc_id)
    try:
        text=fake_parsing(doc_id)
    except ConnectionError as e:
        logger.exception('문서 파싱 실패: %s', doc_id)
        raise ExternalServiceError('문서 변환 서비스에 연결하지 못했음', detail=str(e)) from e

    logger.info('적재 완료: %s', doc_id)
    return text

In [8]:
ingest('DOC-HR-001')

14:46:47 INFO     app.services.document: 적재 시작: DOC-HR-001
14:46:47 INFO     app.services.document: 적재 완료: DOC-HR-001


'DOC-HR-001의 본문 텍스트'

In [ ]:
try:
    ingest('DOC-BAD')
except AgentError as e:
    print('Agent Error')
    print(f'status_code: {e.status_code}')
    print(f'code: {e.code}')
    print(f'message: {e.message}')
    print(f'detail: {e.detail}')